# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Title: {getattr(metadata, 'name', '')}\nDescription: {getattr(metadata, 'description', '')}\n\nSchema URL: {croissant_url}")

## 2. Data Overview
Review available record sets and their corresponding field and column `@id` values. All references to entities use their `@id` for consistency.

In [ ]:
# Extract and examine record sets, listing their @id, fields, and columns.
record_sets = list(dataset.record_sets)
if not record_sets:
    print("No record sets found in the dataset (the dataset may not contain structured tabular data exposed as record sets).\n")
else:
    for rs in record_sets:
        print(f"Record Set @id: {rs.id}")
        # List columns and fields with their @id
        column_ids = [c.id for c in getattr(rs, 'columns', [])]
        field_ids = [f.id for f in getattr(rs, 'fields', [])]
        if column_ids:
            print(f"  Columns: {column_ids}")
        if field_ids:
            print(f"  Fields: {field_ids}")
        print("-"*40)
if not record_sets:
    print("Please refer to the Croissant schema manually or see available metadata for further insights.")


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

> **Note**: If the above produced no record sets, refer to the documentation or examine the dataset's files via the metadata for unstructured or alternative data sources.

In [ ]:
# If record sets exist, extract records to DataFrames by @id
dataframes = {}
if not record_sets:
    print("No record sets available for extraction.")
else:
    for rs in record_sets:
        print(f"Loading records for Record Set '{rs.id}'...")
        try:
            records = list(dataset.records(record_set=rs.id))
            if records:
                df = pd.DataFrame(records)
                dataframes[rs.id] = df
                print(f"  DataFrame columns: {df.columns.tolist()}")
                print(f"  Number of records: {len(df)}")
            else:
                print("  No records found in this record set.")
        except Exception as e:
            print(f"  Error extracting records: {e}")
    print("\nAvailable DataFrames by Record Set @id:")
    for k in dataframes.keys():
        print(f"  {k}")
    # Show preview for the first available record set
    if dataframes:
        primary_rs_id = list(dataframes.keys())[0]
        print("\nPreview for DataFrame from Record Set @id:", primary_rs_id)
        display(dataframes[primary_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps & transformations. All fields are referenced by their `@id`.

In [ ]:
# Example EDA: filtering and normalization for a numeric field
import numpy as np

if dataframes:
    # Select the primary record set for demonstration
    primary_rs_id = list(dataframes.keys())[0]
    df = dataframes[primary_rs_id]
    # Try to detect a numeric field
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_candidates:
        print("No numeric fields detected for EDA.")
    else:
        numeric_field_id = numeric_candidates[0]   # Use the first detected numeric field
        print(f"Using numeric field for analysis (by @id): {numeric_field_id}")
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].mean()>0 else 10
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())
        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Optional: group by the first non-numeric column
        group_candidates = [col for col in df.columns if col != numeric_field_id]
        group_field_id = None
        for col in group_candidates:
            if df[col].dtype==object:
                group_field_id = col
                break
        if group_field_id:
            grouped_df = (
                filtered_df.groupby(group_field_id)[numeric_field_id]
                .mean()
                .reset_index()
                .sort_values(numeric_field_id, ascending=False)
            )
            print(f"Grouped mean '{numeric_field_id}' by '{group_field_id}':")
            display(grouped_df.head())
        else:
            print("No suitable non-numeric group field found for grouping.")
else:
    print("No DataFrame available for EDA. Check if your dataset provides structured data.")

## 5. Visualization
Visualize data distributions or relationships between fields using matplotlib or pandas plotting.

In [ ]:
import matplotlib.pyplot as plt

if dataframes and 'numeric_field_id' in locals():
    primary_rs_id = list(dataframes.keys())[0]
    df = dataframes[primary_rs_id]

    plt.figure(figsize=(8, 4))
    df[numeric_field_id].hist(bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If a group_field_id is available, plot mean per category
    if 'group_field_id' in locals() and group_field_id:
        grouped = df.groupby(group_field_id)[numeric_field_id].mean()
        grouped.sort_values(ascending=False).plot(
            kind='bar', figsize=(10, 4), title=f'Mean {numeric_field_id} by {group_field_id}'
        )
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()
else:
    print("Visualization skipped: no numeric field detected in extracted data.")

## 6. Conclusion
This notebook demonstrated how to load, review, and analyze a Croissant-described dataset using the `mlcroissant` library.

* You learned to load a dataset from a Croissant schema URL and print its metadata.
* You explored any available record sets and identified fields and columns by their `@id`s.
* You extracted and analyzed tabular data (if present), including filtering, normalization, grouping, and basic visualization.

For more advanced usage and details, refer to the [mlcroissant documentation](https://mlcommons.org/croissant/) and examine your dataset's specific schema for further processing possibilities.